<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Data Interoperability with Unity Catalog</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">2. Working with Managed Tables in Unity Catalog</span>
    </div>
</div>

<p style="font-size: 1em; text-align: center; line-height: 0; padding-top: 9px; margin: 4px 0">
<img
src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
alt="Databricks Learning"
>
</div>

# 2.3 Lecture: External Access to Managed Tables

This lesson covers the two access patterns for UC managed tables from outside Databricks - the Unity Catalog REST API and the Iceberg REST Catalog - and the credential vending mechanism that underpins both, along with the steps to set up external client access.

## Learning Objectives

By the end of this lesson, you will be able to:
- Describe the two external access patterns for UC managed tables (UC REST API and Iceberg REST Catalog)
- Explain how credential vending underpins both access patterns and is the same mechanism UC uses internally
- Explain the role and capabilities of the Iceberg REST Catalog
- Outline the steps for setting up external client access to UC tables

## A. Providing Access to Managed Tables

Unity Catalog exposes two open access patterns that together open managed tables to a wide range of external engines without giving up governance.

### Underpinning the Open Lakehouse

The Unity REST API serves Delta clients, and the Iceberg REST Catalog serves Iceberg-compatible clients. Both are open specifications, so external engines can read (and in many cases write) UC-managed tables using the integration that fits their native table format.
<br/>
<br/>
<div style="font-size: 1em; display: flex; align-items: center; justify-content: center; gap: 8px; flex-wrap: wrap; overflow-x: auto; max-width: 100%; margin: 16px 0;">
  <div style="background: #FF3621; border: 2px solid #CC2B1A; color: #fff; padding: 16px 22px; border-radius: 6px; text-align: center; font-weight: bold;">Unity Catalog<br/><span style="font-weight: normal; font-size: 0.85em;">Managed Tables</span></div>
  <div style="color: #999; font-size: 1.3em;">&#10132;</div>
  <div style="display: flex; flex-direction: column; gap: 10px;">
    <div style="display: flex; align-items: center; gap: 8px;">
      <div style="background: #e3f2fd; border: 2px solid #1976d2; padding: 12px 16px; border-radius: 6px; text-align: center; width: 200px;"><strong>Unity REST API</strong><br/><span style="font-size: 0.85em; color: #555;">Delta-native access<br/>(read / write / create)</span></div>
      <div style="color: #999; font-size: 1.3em;">&#10132;</div>
      <div style="background: #e8f5e9; border: 2px solid #4caf50; padding: 12px 16px; border-radius: 6px; text-align: center; width: 200px;"><strong>Delta Clients</strong><br/><span style="font-size: 0.85em; color: #555;">Apache Spark, DuckDB,<br/>Daft, Microsoft Fabric</span></div>
    </div>
    <div style="display: flex; align-items: center; gap: 8px;">
      <div style="background: #e3f2fd; border: 2px solid #1976d2; padding: 12px 16px; border-radius: 6px; text-align: center; width: 200px;"><strong>Iceberg REST Catalog</strong><br/><span style="font-size: 0.85em; color: #555;">Iceberg-spec access<br/>(read / write / create)</span></div>
      <div style="color: #999; font-size: 1.3em;">&#10132;</div>
      <div style="background: #fff3e0; border: 2px solid #e65100; padding: 12px 16px; border-radius: 6px; text-align: center; width: 200px;"><strong>Iceberg Clients</strong><br/><span style="font-size: 0.85em; color: #555;">Snowflake, Trino,<br/>Apache Flink, Dremio</span></div>
    </div>
  </div>
</div>

### What Each Pattern Is For

The **Unity REST API** is the access path for Delta-native clients - external Apache Spark with the Unity Catalog Spark connector, plus other Delta-aware engines like DuckDB, Daft, and Microsoft Fabric. Endpoint is `/api/2.1/unity-catalog` on the workspace URL, and it supports read, write, and create against managed and external Unity Catalog managed tables.

The **Iceberg REST Catalog (IRC)** is the access path for any engine that speaks the Apache Iceberg REST spec - Snowflake, Trino, Dremio, Flink, Spark with the Iceberg connector, PyIceberg, and others. It supports read, write, and create against managed Iceberg tables, and read-only access to UC managed tables that have UniForm enabled.

Apache Spark sits in both groups in practice: pick the Unity REST API if you're working with UC managed tables and the Iceberg REST Catalog if you're working with Iceberg tables. Credential vending is the underlying mechanism that issues short-lived, path-scoped storage credentials for both APIs - covered separately later.

## B. Unity Catalog API

The UC REST API is the primary surface for programmatic interaction with Unity Catalog, covering metadata management, permissions, and lineage.

### Programmatic Access to Metadata and Governance

The table below summarizes the capabilities exposed by the API and the common ways teams use it.

| Area | Capabilities |
|------|-------------|
| **Coverage** | Table, schema, and catalog management; Permission and access control; Metadata exploration; Data lineage and audit logs |
| **Authentication** | OAuth 2.0 and service principals; Personal access tokens; AWS IAM / Azure AD integration |
| **Use Cases** | Custom data catalog applications; DevOps automation; Third-party tool integration; Metadata-driven workflow orchestration |
| **Access Methods** | CLI, SDK, REST API |

<div style="font-size: 1em; border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #0d47a1; font-size: 1.1em;">Open Source</strong>
            <p style="margin: 8px 0 0 0; color: #333;">The Unity Catalog API specification is open source: <a href="https://github.com/unitycatalog/unitycatalog/tree/main/api" style="color:#1565c0;">github.com/unitycatalog/unitycatalog</a></p>
        </div>
    </div>
</div>

## C. What is the Iceberg REST Catalog?

The Iceberg REST Catalog standardizes how engines discover and read Iceberg tables, and Unity Catalog implements this specification directly.

### Gateway to Interoperability

The Iceberg REST Catalog is a **centralized directory service** that provides metadata and access to Iceberg tables through a standardized API.
<br/><br/>

<div style="display: flex; align-items: stretch; justify-content: center; gap: 0; padding: 24px 16px; max-width: 1200px; margin: 16px auto; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; font-size: 1em"><div style="border: 2px dashed #aaa; border-radius: 8px; padding: 24px 16px 16px 16px; position: relative; background: #fafafa; display: flex; align-items: center"><div style="position: absolute; top: -10px; left: 16px; background: #fafafa; padding: 0 8px; font-size: 1em; font-weight: 600; color: #555">Table Creation</div><div style="width: 180px; background: #fff; border: 2px solid #888; border-radius: 6px; padding: 14px 18px; font-weight: 600; text-align: center; line-height: 1.4"><div>Apache Spark</div><div style="font-weight: 400; color: #555; font-size: 1em">creates table</div></div></div><div style="display: flex; align-items: center; align-self: center; padding: 0 16px"><div style="width: 60px; height: 2px; background: #555"></div><div style="width: 0; height: 0; border-top: 6px solid transparent; border-bottom: 6px solid transparent; border-left: 9px solid #555"></div></div><div style="border: 2px solid #FF3621; border-radius: 8px; padding: 24px 16px 16px 16px; position: relative; background: rgba(255, 54, 33, 0.08); display: flex; align-items: center"><div style="position: absolute; top: -10px; left: 16px; background: #fff; padding: 0 8px; font-size: 1em; font-weight: 600; color: #CC2B1A">Iceberg REST Catalog</div><div style="width: 180px; background: #fff; border: 2px solid #888; border-radius: 6px; padding: 14px 18px; font-weight: 600; text-align: center; line-height: 1.4"><div>Unity Catalog</div><div style="font-weight: 400; color: #555; font-size: 1em">implements spec</div></div></div><div style="display: flex; align-items: center; align-self: center; padding: 0 16px"><div style="width: 60px; height: 2px; background: #555"></div><div style="width: 0; height: 0; border-top: 6px solid transparent; border-bottom: 6px solid transparent; border-left: 9px solid #555"></div></div><div style="border: 2px dashed #aaa; border-radius: 8px; padding: 24px 16px 16px 16px; position: relative; background: #fafafa; display: flex; flex-direction: column; gap: 10px"><div style="position: absolute; top: -10px; left: 16px; background: #fafafa; padding: 0 8px; font-size: 1em; font-weight: 600; color: #555">Table Access</div><div style="width: 180px; background: #fff; border: 2px solid #888; border-radius: 6px; padding: 12px 18px; font-weight: 600; text-align: center; line-height: 1.4"><div>Trino</div><div style="font-weight: 400; color: #555; font-size: 1em">sees it</div></div><div style="width: 180px; background: #fff; border: 2px solid #888; border-radius: 6px; padding: 12px 18px; font-weight: 600; text-align: center; line-height: 1.4"><div>Flink</div><div style="font-weight: 400; color: #555; font-size: 1em">processes it</div></div><div style="width: 180px; background: #fff; border: 2px solid #888; border-radius: 6px; padding: 12px 18px; font-weight: 600; text-align: center; line-height: 1.4"><div>Snowflake</div><div style="font-weight: 400; color: #555; font-size: 1em">queries it</div></div></div></div>

**Key Benefits:**
- Standardized API that any compute engine can use
- True interoperability without data duplication
- Unity Catalog provides a full implementation of the spec

## D. Iceberg REST Catalog Integration

A wide range of analytics engines can connect to UC through the Iceberg REST Catalog, with read or read/write capabilities depending on the engine.

### External Engines That Can Connect

The matrix below lists the engines most commonly used with UC and the level of access each one supports.

| Engine | Access Type | Capabilities |
|--------|-----------|-------------|
| **Snowflake** | Read | Query UC tables as Iceberg tables via catalog integration |
| **Trino / Presto** | Read/Write | Full SQL access to UC managed tables |
| **Apache Flink** | Read/Write | Streaming and batch processing of UC tables |
| **Iceberg Kafka Connect** | Write | Stream data directly into UC managed Iceberg tables |
| **Apache Spark** | Read/Write | Native Iceberg catalog access from external Spark clusters |
| **PyIceberg** | Read | Python client for programmatic table access |

<div style="font-size: 1em; border-left: 4px solid #7b1fa2; background: #f3e5f5; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #4a148c; font-size: 1.1em;">Specification</strong>
            <p style="margin: 8px 0 0 0; color: #333;">The Iceberg REST Catalog is based on the open Apache Iceberg REST spec: <a href="https://github.com/apache/iceberg/blob/main/open-api/rest-catalog-open-api.yaml" style="color:#4a148c;">REST Catalog Open API</a></p>
        </div>
    </div>
</div>

## E. Credential Vending - the Underlying Mechanism

<p style="font-size: 1em; line-height: 1.6; color: #333">Credential vending is not a third access pattern - it is the underlying mechanism that both the UC REST API and the Iceberg REST Catalog use to issue short-lived, scoped credentials to external clients on demand, which keeps governance enforced even when compute lives outside Databricks. The Iceberg REST spec carries vended credentials directly in the <code>loadTable</code> response (in the <code>config</code> / <code>storage-credentials</code> fields), so every Iceberg REST call is already a credential-vending call.</p>

This is the same mechanism UC uses internally when a Databricks cluster reads a managed table - external access just exposes the existing machinery to non-Databricks engines. Open Sharing is a separate sibling protocol with its own short-lived-access design (the sharing server pre-signs URLs to data files rather than vending storage credentials to the client), so it is governed by UC but does not use credential vending.

The pull-down below shows the credential-vending handshake step by step - how an external client authenticates to Unity Catalog, gets authorized at the table level, has the access decision recorded for audit, and receives a short-lived, path-scoped credential to read or write data files directly against cloud storage.


<br/>
<details>
  <summary style="cursor: pointer; list-style: none; user-select: none">
    <div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
      <div style="display: flex; align-items: center; gap: 12px">
        <span>&#x25B6;</span>
        <strong style="font-size: 1.1em; color: #1B5162">Show Sequence Diagram</strong>
      </div>
    </div>
  </summary>

  <div class="mermaid" id="diagram-2-3-credential-vending-seq" style="display: none; font-size: 1em;">
sequenceDiagram
    autonumber
    participant Client as External Client<br/>(Snowflake / EMR / PyIceberg)
    participant UC as Unity Catalog<br/>(Iceberg REST + Token Service)
    participant CSP as Cloud Storage<br/>(S3 / ADLS / GCS)
    Client->>UC: 1. Authenticate (OAuth token / SP credentials)
    UC-->>Client: 2. Issue UC API token
    Client->>UC: 3. loadTable() against Iceberg REST endpoint
    UC->>UC: 4. Authorize: SELECT or MODIFY on table?
    UC->>UC: 5. Write audit record (caller, table, action, scope)
    UC-->>Client: 6. Iceberg metadata + scoped, short-lived storage credential
    Client->>CSP: 7. Read/write data files using vended credential
    CSP-->>Client: 8. Data
    Note over UC,CSP: Vended credential expires (default ~1h)<br/>and is scoped to specific paths only.<br/>Audit is captured by UC at step 5 (control plane),<br/>not at step 7 (data plane).
  </div>

</details>

<script type="module">
import mermaid from "https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs";
mermaid.initialize({ startOnLoad: false });

const details = document.querySelector("details");
const node = document.getElementById("diagram-2-3-credential-vending-seq");
if (details && node) {
  let rendered = false;
  details.addEventListener("toggle", async () => {
    if (!details.open || rendered) return;
    rendered = true;
    node.style.display = "block";
    try {
      await mermaid.run({ querySelector: "#diagram-2-3-credential-vending-seq" });
    } catch(e) {
      await new Promise(r => setTimeout(r, 1000));
      await mermaid.run({ querySelector: "#diagram-2-3-credential-vending-seq" });
    }
    node.querySelectorAll('svg text, svg .nodeLabel, svg foreignObject div, svg span').forEach(el => { el.style.fontSize = '1em'; });
    node.querySelectorAll('svg .note text, svg .noteText, svg g.note text').forEach(el => { el.style.fontSize = '0.85em'; });
  });
}
</script>

<div style="font-size: 1em; border-left: 4px solid #ff9800; background: #fff3e0; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #e65100; font-size: 1.1em;">Security Best Practice</strong>
            <p style="margin: 8px 0 0 0; color: #333;">Always use credential vending with time-limited tokens rather than long-lived credentials. Configure IP-based restrictions and audit all external access patterns.</p>
        </div>
    </div>
</div>

The capabilities below describe the security and audit guarantees credential vending provides.

| Aspect | Detail |
|--------|--------|
| **Mechanism** | Generates temporary, scoped credentials for external access |
| **Governance** | Maintains UC governance while enabling external access |
| **Security Controls** | Fine-grained access scopes, token expiration and rotation, IP-based restrictions |
| **Audit** | Comprehensive audit logging of all external access |

## F. Setting Up External Access

<p style="font-size: 1em; line-height: 1.6; color: #333">The four-step sequence below is the typical configuration path for granting an external engine access to UC managed tables. The pull-down has the detail behind each step.</p>

<div style="font-size: 1em; display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; margin: 16px 0">

  <div style="background: linear-gradient(135deg, #E3F2FD 0%, #BBDEFB 100%); border: 1px solid #1976D2; border-radius: 10px; padding: 16px">
    <div style="font-weight: bold; color: #0D47A1; margin-bottom: 6px">1. Enable External Access</div>
    <div style="color: #1B3139">Open the doors at workspace and metastore level.</div>
  </div>

  <div style="background: linear-gradient(135deg, #E8F5E9 0%, #C8E6C9 100%); border: 1px solid #2E7D32; border-radius: 10px; padding: 16px">
    <div style="font-weight: bold; color: #1B5E20; margin-bottom: 6px">2. Configure Authentication</div>
    <div style="color: #1B3139">Identify and trust the external caller.</div>
  </div>

  <div style="background: linear-gradient(135deg, #FFF3E0 0%, #FFE0B2 100%); border: 1px solid #E65100; border-radius: 10px; padding: 16px">
    <div style="font-weight: bold; color: #BF360C; margin-bottom: 6px">3. External Client Setup</div>
    <div style="color: #1B3139">Wire the external engine to UC.</div>
  </div>

  <div style="background: linear-gradient(135deg, #F3E5F5 0%, #E1BEE7 100%); border: 1px solid #6A1B9A; border-radius: 10px; padding: 16px">
    <div style="font-weight: bold; color: #4A148C; margin-bottom: 6px">4. Monitor and Manage</div>
    <div style="color: #1B3139">Verify, audit, and rotate credentials.</div>
  </div>

</div>


<br/>
<details>
  <summary style="cursor: pointer; list-style: none; user-select: none">
    <div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
      <div style="display: flex; align-items: center; gap: 12px">
        <span>&#x25B6;</span>
        <strong style="font-size: 1.1em; color: #1B5162">Expand for More Details</strong>
      </div>
    </div>
  </summary>
  <div style="border-left: 4px solid #1B5162; background: transparent; padding: 0 20px 16px 20px; border-radius: 0 0 4px 4px; margin: -16px 0 16px 0">
    <div style="display: flex; align-items: flex-start; gap: 12px">
      <span style="visibility: hidden">&#x25B6;</span>
      <div>
        <strong style="color: #1B5162">Step 1 - Enable External Access</strong>
        <ul style="line-height: 1.8; color: #333; margin: 6px 0 14px 0">
          <li>A workspace and metastore-level toggle exposes the Unity Catalog Iceberg REST endpoint along with any network or IP-based access policies the platform team requires.</li>
          <li>Turning this on makes <code>/api/2.1/unity-catalog/iceberg-rest</code> reachable so external clients can discover and load UC tables, with allow-lists applied before any external traffic flows.</li>
          <li>This step is performed once by a platform or governance owner in the workspace admin console and Unity Catalog metastore settings.</li>
        </ul>
        <strong style="color: #1B5162">Step 2 - Configure Authentication</strong>
        <ul style="line-height: 1.8; color: #333; margin: 6px 0 14px 0">
          <li>An OAuth service principal (or a per-user token for dev/test) is the identity external engines authenticate as, with UC grants and any IP allow-listing applied to it.</li>
          <li>Unity Catalog issues short-lived access tokens scoped to whatever the service principal has been granted, and credential-vends storage credentials only for the specific files needed.</li>
          <li>This is set up in Unity Catalog (service principal creation and grants) and in the workspace secret scope used by the external engine to store the SP secret.</li>
        </ul>
        <strong style="color: #1B5162">Step 3 - External Client Setup</strong>
        <ul style="line-height: 1.8; color: #333; margin: 6px 0 14px 0">
          <li>The external engine is configured to point at UC: the Iceberg REST endpoint URL, the service principal credentials, the warehouse or catalog name, and engine-specific options.</li>
          <li>Once configured, the external engine can discover, load, and (with <code>MODIFY</code> granted) write UC tables as if UC were a native Iceberg catalog.</li>
          <li>The exact mechanism is engine-specific - Snowflake calls this a <code>CATALOG INTEGRATION</code>, EMR uses Spark catalog config, PyIceberg uses <code>load_catalog("rest", ...)</code>.</li>
        </ul>
        <strong style="color: #1B5162">Step 4 - Monitor and Manage</strong>
        <ul style="line-height: 1.8; color: #333; margin: 6px 0 14px 0">
          <li>Day-2 operations include audit log review, system-table queries, secret rotation, and pushdown verification.</li>
          <li><code>system.access.audit</code> surfaces who touched what, and <code>system.query.history</code> shows what queries the external engine ran against UC.</li>
          <li>Service principal secrets should be rotated on a cadence and grants adjusted as access needs change.</li>
        </ul>
      </div>
    </div>
  </div>
</details>

<div style="font-size: 1em; border-left: 4px solid #ffc107; background: #fffde7; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
    <div style="display: flex; align-items: flex-start; gap: 12px">
        <div>
            <strong style="color: #ff8f00; font-size: 1.1em">Coming Up</strong>
            <p style="margin: 8px 0 0 0; color: #333">Section 3 walks through these steps with hands-on demos for Snowflake, EMR, and other external systems.</p>
        </div>
    </div>
</div>

## Key Takeaways

UC exposes two access patterns for external engines, both governed by the same underlying mechanism.

- The **UC REST API** provides programmatic access to metadata, governance, and lineage
- The **Iceberg REST Catalog** lets any Iceberg-compatible engine discover and read/write UC tables via the open spec
- **Credential vending** is the underlying mechanism behind both - it issues short-lived, path-scoped storage credentials on every table access, and is the same mechanism UC uses internally for Databricks compute
- Open Sharing is a separate sibling protocol governed by UC but with its own short-lived-access design (pre-signed URLs), distinct from credential vending
- External access maintains **full UC governance** including permissions, audit, and lineage
- Setup follows a 4-step process: Enable -> Authenticate -> Connect -> Monitor

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>